# Day 39 Practice: Build, Debug and Test Toolbox

Most of today is for **building your capstone**. Exercise 1 is the warm-up for everyone. Exercises 2–6 are a
toolbox: do the one that matches what you are stuck on, or work through them when your MVP is done.

| If you are stuck on… | Do |
|----------------------|----|
| writing tests | Exercises 2 and 3 |
| a long, messy function | Exercise 4 |
| validating user input | Exercise 5 |
| loading/saving safely | Exercise 6 (challenge) |

✅ Exercise 1: everyone · Exercises 2–5: toolbox · Exercise 6: challenge

In [ ]:
import json
from pathlib import Path

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)

## Exercise 1: Read the traceback

Mei's flashcard app crashed when she chose "statistics":

```
Traceback (most recent call last):
  File "/home/mei/flashcards/main.py", line 58, in <module>
    main()
  File "/home/mei/flashcards/main.py", line 49, in main
    show_stats(deck)
  File "/home/mei/flashcards/main.py", line 31, in show_stats
    print(f"{card.front}: {accuracy(card.times_right, card.times_wrong):.1f}%")
  File "/home/mei/flashcards/models.py", line 22, in accuracy
    return times_right / (times_right + times_wrong) * 100
ZeroDivisionError: division by zero
```

**a)** In which file, line and function did the error actually happen?
**b)** Which values of `times_right` and `times_wrong` cause it? What kind of card is that?
**c)** What *should* the accuracy of such a card be? Decide, then write the fixed `accuracy` below with two
`assert`s (a normal card and the problem card).

*Your answers:* a) … b) … c) …

In [ ]:
# your code here

## Exercise 2: Write tests for working code

These two functions from a Habit Tracker are correct. Write the tests: at least **two asserts per function**,
including an **edge case** (empty list, all `False`, exactly the maximum), and for `completion_rate` an **error
case**: check that it raises `ValueError` using `try/except` (in a pytest file you would use `pytest.raises`).

In [ ]:
def completion_rate(done_days, total_days):
    """Return the % of days a habit was done, rounded to a whole number."""
    if total_days <= 0 or done_days < 0 or done_days > total_days:
        raise ValueError("done_days must be between 0 and total_days, and total_days must be positive.")
    return round(done_days / total_days * 100)


def longest_streak(days):
    """Return the longest run of True values in a list such as [True, True, False, True]."""
    best = 0
    current = 0
    for done in days:
        if done:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best


def test_completion_rate():
    pass  # your asserts here


def test_longest_streak():
    pass  # your asserts here


test_completion_rate()
test_longest_streak()
print("tests ran")

## Exercise 3: Find the bugs with tests

The library functions below have **three** bugs. The checks in the next cell know the right answers.

1. Run the checks cell (it only defines `run_checks`).
2. Copy the buggy code into the empty cell, run it, then call `run_checks()` and read which `assert` fails.
3. Fix one bug at a time until `run_checks()` prints `all checks passed`.

```python
def books_on_loan(books):
    """Return the titles of the books that are on loan (borrower is not None)."""
    titles = []
    for book in books:
        if book["borrower"] is None:
            titles.append(book["title"])
    return titles


def count_loans(books, member):
    """Return how many books this member has borrowed (upper/lower case does not matter)."""
    count = 0
    for book in books:
        if book["borrower"] == member:
            count += 1
    return count


def can_borrow(books, member, limit=3):
    """Return True if the member has fewer than `limit` books."""
    return count_loans(books, member) <= limit
```

In [ ]:
def run_checks():
    """Check the three library functions on a tiny example."""
    books = [
        {"title": "The Silent River", "borrower": "Omar"},
        {"title": "Learning to Code", "borrower": None},
        {"title": "Cats, Dogs and Other Friends", "borrower": "omar"},
        {"title": "Night Trains", "borrower": "Zara"},
    ]
    assert books_on_loan(books) == ["The Silent River", "Cats, Dogs and Other Friends", "Night Trains"]
    assert count_loans(books, "Omar") == 2
    assert count_loans(books, "Mei") == 0
    assert can_borrow(books, "Zara")
    assert not can_borrow(books, "Omar", limit=2)
    print("all checks passed")

In [ ]:
# paste the buggy code here, then fix it and call run_checks()

## Exercise 4: Refactor a long function

`habit_report` works but does everything in one place. Split it into small functions, for example
`days_done(habit, week)`, `week_marks(habit, week)` (the `X..X...` string) and `best_habit(habits, week)`, and a new
`habit_report` that uses them. Finish with `assert` that the new output equals `original`.

Expected output (from the original):
```
WEEKLY HABIT REPORT
--------------------------------
Read 20 pages     XX.X.XX   71%
Walk 30 minutes   X.X....   29%
Practise Python   XXXX.XX   86%
--------------------------------
Best habit: Practise Python
```

In [ ]:
week = ["2026-12-01", "2026-12-02", "2026-12-03", "2026-12-04", "2026-12-05", "2026-12-06", "2026-12-07"]
habits = [
    {"name": "Read 20 pages", "done_dates": ["2026-12-01", "2026-12-02", "2026-12-04", "2026-12-06", "2026-12-07"]},
    {"name": "Walk 30 minutes", "done_dates": ["2026-12-01", "2026-12-03"]},
    {"name": "Practise Python", "done_dates": ["2026-12-01", "2026-12-02", "2026-12-03", "2026-12-04",
                                               "2026-12-06", "2026-12-07"]},
]


def habit_report(habits, week):
    """Return a weekly report for all habits."""
    lines = []
    lines.append("WEEKLY HABIT REPORT")
    lines.append("-" * 32)
    best_name = ""
    best_count = -1
    for habit in habits:
        count = 0
        for day in week:
            if day in habit["done_dates"]:
                count += 1
        percent = round(count / len(week) * 100)
        marks = ""
        for day in week:
            if day in habit["done_dates"]:
                marks += "X"
            else:
                marks += "."
        lines.append(f"{habit['name']:<18}{marks}  {percent:>3}%")
        if count > best_count:
            best_count = count
            best_name = habit["name"]
    lines.append("-" * 32)
    lines.append(f"Best habit: {best_name}")
    return "\n".join(lines)


original = habit_report(habits, week)
print(original)

In [ ]:
# your code here

## Exercise 5: Validate a date

Many capstones store dates as text `YYYY-MM-DD`. Write `parse_date(text)` that returns the date text without
extra spaces, or raises `ValueError` with a helpful message when:
- it is not three parts separated by `-`, or the parts are not all digits;
- the year does not have 4 digits, or the month/day do not have 2 digits;
- the month is not 1–12 or the day is not 1–31 (we accept 31 February; perfect checking is a Could).

Test it with `assert` for good dates and with `try/except` for at least four bad ones: `"2026/12/01"`,
`"2026-13-01"`, `"26-12-01"`, `"2026-12-1x"`.

In [ ]:
# your code here

## Exercise 6 (challenge): Safe load and save

Write `save_json(data, path)` and `load_list(path)`:
- a missing file returns `[]`;
- a damaged file (not valid JSON) returns `[]` and prints a warning;
- a file that contains valid JSON but **not a list** also returns `[]` with a warning.

Test all four situations (good file, missing, damaged, not a list) by writing small files into `output/`.

In [ ]:
# your code here